# PyTorch Re-run for Fair Comparison

The Week 2 notebook did not record training time, MAE or R-squared, and had no fixed seed.
This notebook repeats the same pipeline (same data, split, scaling, architecture, loss,
optimizer, batch size and epochs) with a fixed seed, a timer, and the same metrics as the
Keras notebook. Test loss is computed after every epoch (as Keras did with validation_data)
so the timing is comparable. The Week 2 notebook is unchanged.

In [1]:
import time
import json
import platform
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

df = fetch_california_housing(as_frame=True).frame

X = df.drop(columns=["MedHouseVal"]).values
y = df["MedHouseVal"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = HousingDataset(X_train_t, y_train_t)
test_dataset = HousingDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Torch version:", torch.__version__)
print("Train:", X_train_t.shape, y_train_t.shape)
print("Test: ", X_test_t.shape, y_test_t.shape)

Torch version: 2.14.0+cpu
Train: torch.Size([16512, 8]) torch.Size([16512, 1])
Test:  torch.Size([4128, 8]) torch.Size([4128, 1])


In [2]:
torch.manual_seed(42)   # seed right before model creation and shuffling

class HousingNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.model(x)

model = HousingNN(input_dim=X_train_t.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50
train_losses, test_losses = [], []

start_time = time.perf_counter()
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    train_losses.append(running_loss / len(train_dataset))

    model.eval()
    test_running = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            test_running += criterion(model(batch_X), batch_y).item() * batch_X.size(0)
    test_losses.append(test_running / len(test_dataset))

training_time = time.perf_counter() - start_time

for epoch in range(4, num_epochs, 5):
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_losses[epoch]:.4f} "
          f"- Test Loss: {test_losses[epoch]:.4f}")
print(f"\nParameters: {sum(p.numel() for p in model.parameters())}")
print(f"Training time: {training_time:.1f} seconds")

Epoch 5/50 - Train Loss: 0.3689 - Test Loss: 0.3715
Epoch 10/50 - Train Loss: 0.3251 - Test Loss: 0.3360
Epoch 15/50 - Train Loss: 0.3034 - Test Loss: 0.3093
Epoch 20/50 - Train Loss: 0.2939 - Test Loss: 0.3040
Epoch 25/50 - Train Loss: 0.2855 - Test Loss: 0.2972
Epoch 30/50 - Train Loss: 0.2810 - Test Loss: 0.3131
Epoch 35/50 - Train Loss: 0.2805 - Test Loss: 0.2944
Epoch 40/50 - Train Loss: 0.2733 - Test Loss: 0.3007
Epoch 45/50 - Train Loss: 0.2721 - Test Loss: 0.2880
Epoch 50/50 - Train Loss: 0.2686 - Test Loss: 0.2877

Parameters: 833
Training time: 39.0 seconds


## Evaluation
Same metrics as the Keras notebook: MSE, MAE, RMSE and R-squared, computed with the
final weights. Results are saved to `results/pytorch_results.json`.

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

model.eval()
with torch.no_grad():
    train_pred = model(X_train_t).squeeze().numpy()
    test_pred = model(X_test_t).squeeze().numpy()

train_mse = mean_squared_error(y_train, train_pred)
train_mae = mean_absolute_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print(f"Train MSE: {train_mse:.4f} | Train MAE: {train_mae:.4f}")
print(f"Test  MSE: {test_mse:.4f} | Test  MAE: {test_mae:.4f}")
print(f"Test  RMSE: {np.sqrt(test_mse):.4f}")
print(f"Test  R-squared: {test_r2:.4f}")

results = {
    "framework": "PyTorch (re-run, seed 42)",
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "device": "CPU",
    "dataset": "California Housing",
    "train_samples": int(X_train_t.shape[0]),
    "test_samples": int(X_test_t.shape[0]),
    "epochs": num_epochs,
    "batch_size": 32,
    "learning_rate": 0.001,
    "random_seed": 42,
    "parameters": int(sum(p.numel() for p in model.parameters())),
    "train_mse": round(float(train_mse), 4),
    "train_mae": round(float(train_mae), 4),
    "test_mse": round(float(test_mse), 4),
    "test_mae": round(float(test_mae), 4),
    "test_rmse": round(float(np.sqrt(test_mse)), 4),
    "test_r2": round(float(test_r2), 4),
    "final_epoch_running_train_loss": round(float(train_losses[-1]), 4),
    "training_time_seconds": round(training_time, 1),
}

with open("../results/pytorch_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved to ../results/pytorch_results.json")

Train MSE: 0.2697 | Train MAE: 0.3505
Test  MSE: 0.2877 | Test  MAE: 0.3597
Test  RMSE: 0.5364
Test  R-squared: 0.7805

Saved to ../results/pytorch_results.json
